# Phase 2: Attention Calibration Distillation — SmolLM2-1.7B-Instruct

**Architecture change**: SmolLM2 uses `LlamaForCausalLM`, which is completely different from GPT-2:
- Separate `q_proj`, `k_proj`, `v_proj` linear layers (not a fused `c_attn`)
- Rotary Position Embeddings (RoPE) applied to Q and K before score computation
- Output projection is `o_proj`, not `c_proj`
- Layers live at `model.model.layers[i].self_attn`, not `model.transformer.h[i].attn`

**All previous fixes retained**: domain-matched demos, averaged teacher targets, combined MSE+KL+L2 loss, gradient clipping.

In [1]:
!pip install transformers torch

## Step 1 — Setup

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# SmolLM2-1.7B-Instruct — instruction-tuned, reliably follows classification format
# bfloat16 = ~3.4 GB per model; two models = ~6.8 GB, well within T4 (15 GB)
model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def load_model():
    return AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        attn_implementation="eager",  # Disable SDPA so our custom attention is called
    ).to(device)

teacher_model = load_model()
student_model = load_model()

for model in [teacher_model, student_model]:
    for param in model.parameters():
        param.requires_grad = False
    model.eval()

n_params = sum(p.numel() for p in teacher_model.parameters())
print(f"Parameters per model : {n_params:,}")
print("Both models loaded in bfloat16 and frozen.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Parameters per model : 1,711,376,384
Both models loaded in bfloat16 and frozen.


## Step 2 — Token IDs for Classification Labels

In [3]:
# SmolLM2's tokenizer may encode "Positive"/"Negative" as single or multiple tokens.
# We take the FIRST token — that is what the model predicts at position -1.
POS_ID = tokenizer.encode("Positive", add_special_tokens=False)[0]
NEG_ID = tokenizer.encode("Negative", add_special_tokens=False)[0]

print(f"'Positive' encodes to: {tokenizer.encode('Positive', add_special_tokens=False)} — using ID {POS_ID}")
print(f"'Negative' encodes to: {tokenizer.encode('Negative', add_special_tokens=False)} — using ID {NEG_ID}")

'Positive' encodes to: [31176] — using ID 31176
'Negative' encodes to: [42529] — using ID 42529


## Step 3 — Demonstrations and Dataset

Demonstrations are product reviews (domain-matched to all queries).  
Forward order: Pos, Neg, Pos, Neg → ends **Negative** → recency bias toward Negative.  
Reversed order ends **Positive** → recency bias toward Positive.  
Averaging both cancels the bias.

In [4]:
DEMOS_FORWARD = [
    ("The wireless earbuds have incredible sound quality and the battery lasts all day.", "Positive"),
    ("This phone case cracked after one drop, completely useless for protection.",        "Negative"),
    ("Absolutely love this kitchen knife, perfectly balanced and razor sharp.",           "Positive"),
    ("The charging cable stopped working after a week, very poor quality.",               "Negative"),
]
DEMOS_REVERSED = list(reversed(DEMOS_FORWARD))


def build_teacher_prompt(demos, query_review):
    """Few-shot teacher prompt using the model's chat template."""
    demo_text = ""
    for review, label in demos:
        demo_text += f"Review: {review}\nSentiment: {label}\n\n"

    user_content = (
        "Classify the sentiment of each product review as Positive or Negative. "
        "Respond with exactly one word.\n\n"
        + demo_text
        + f"Review: {query_review}\nSentiment:"
    )
    messages = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def build_student_prompt(query_review):
    """Zero-shot student prompt using the model's chat template."""
    user_content = (
        "Classify the sentiment of the following product review as Positive or Negative. "
        "Respond with exactly one word.\n\n"
        f"Review: {query_review}\nSentiment:"
    )
    messages = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


TRAIN_REVIEWS = [
    ("Perfect fit, excellent material, exactly what was described.",              "Positive"),
    ("Works flawlessly, the best purchase I have made this year.",                "Positive"),
    ("Superb build quality and arrived ahead of schedule.",                       "Positive"),
    ("Very happy with this product, does exactly what it promises.",              "Positive"),
    ("Brilliant item, my whole family absolutely loves it.",                      "Positive"),
    ("Top quality, will definitely be buying this again.",                        "Positive"),
    ("Exceeded all my expectations, incredibly satisfied with the purchase.",     "Positive"),
    ("Outstanding value for money, highly recommend to everyone.",                "Positive"),
    ("Completely fell apart after just three days of light use.",                 "Negative"),
    ("Arrived already broken and the seller refused to help.",                    "Negative"),
    ("Terrible quality, nothing like the photos shown on the listing.",           "Negative"),
    ("Stopped working after one week, total disappointment.",                     "Negative"),
    ("Smells awful and the color was completely different from what was shown.",  "Negative"),
    ("The worst product I have ever bought, avoid at all costs.",                 "Negative"),
    ("Broke immediately on first use, packaging was also damaged.",               "Negative"),
    ("Very flimsy and cheap feeling, not worth even half the price.",             "Negative"),
]

TEST_REVIEWS = [
    ("Absolutely love it, the best thing I have bought all year.",                "Positive"),
    ("Great quality for the price, very impressed with the finish.",              "Positive"),
    ("Exactly what I needed, works like a charm every single time.",              "Positive"),
    ("Fantastic product, arrived quickly and in perfect condition.",              "Positive"),
    ("So happy with this purchase, would buy again without hesitation.",          "Positive"),
    ("Terrible smell and fell apart within a week of use.",                       "Negative"),
    ("Completely useless, broke on the very first day.",                          "Negative"),
    ("Nothing like the description, very misleading product photos.",             "Negative"),
    ("Poor quality and the customer service was incredibly unhelpful.",           "Negative"),
    ("Returned immediately, absolute rubbish, do not waste your money.",          "Negative"),
]

print(f"Train: {len(TRAIN_REVIEWS)} reviews  |  Test: {len(TEST_REVIEWS)} reviews")

Train: 16 reviews  |  Test: 10 reviews


## Step 4 — Capture Baseline Predictions (BEFORE any injection)

We record the untrained student's predictions now, while the model is still completely unmodified.

In [5]:
def predict(model, prompt_str):
    """Single forward pass. Returns (predicted_label, P(Positive), P(Negative))."""
    inputs = tokenizer(prompt_str, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[:, -1, :].float()  # float32 for softmax
        probs  = F.softmax(logits, dim=-1)
    p_pos = probs[0, POS_ID].item()
    p_neg = probs[0, NEG_ID].item()
    label = "Positive" if p_pos > p_neg else "Negative"
    return label, p_pos, p_neg


def predict_teacher_averaged(review):
    """Teacher prediction averaged across both demo orderings to cancel recency bias."""
    _, fp, fn = predict(teacher_model, build_teacher_prompt(DEMOS_FORWARD,  review))
    _, rp, rn = predict(teacher_model, build_teacher_prompt(DEMOS_REVERSED, review))
    avg_p_pos = (fp + rp) / 2
    avg_p_neg = (fn + rn) / 2
    return ("Positive" if avg_p_pos > avg_p_neg else "Negative"), avg_p_pos, avg_p_neg


# Save baseline (untrained student) predictions for all test reviews
print("Capturing baseline (untrained student) predictions...")
baseline_preds = {}
for review, true_label in TEST_REVIEWS:
    label, pp, pn = predict(student_model, build_student_prompt(review))
    baseline_preds[review] = (label, pp, pn)
print("Done. Baseline predictions saved.")

Capturing baseline (untrained student) predictions...
Done. Baseline predictions saved.


## Step 5 — RoPE Utilities

SmolLM2 uses Rotary Position Embeddings. We implement the standard rotation functions here so we are not version-dependent on transformers internals.

In [6]:
def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


def apply_rotary_pos_emb(q, k, cos, sin):
    """
    Apply rotary position embeddings to Q and K.
    cos/sin: (1, 1, T, head_dim) or (B, 1, T, head_dim)
    q, k:    (B, H, T, head_dim)
    """
    # Broadcast across heads
    if cos.dim() == 3:          # (1, T, D) — older API
        cos = cos.unsqueeze(1)  # (1, 1, T, D)
        sin = sin.unsqueeze(1)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


print("RoPE utilities defined.")

RoPE utilities defined.


## Step 6 — CalibratedAttention for LlamaAttention

Rewrites the adapter for the Llama architecture:
- Accesses `q_proj`, `k_proj`, `v_proj` separately
- Applies RoPE to Q and K before computing Delta (same rotated space as baseline S)
- Handles GQA (repeat K, V heads to match Q heads if needed)
- Computes in float32 to maintain gradient precision, casts back for `o_proj`

In [7]:
class CalibratedAttention(nn.Module):
    """
    Wraps a frozen LlamaAttention and injects:
        S' = S + delta,   delta = (Q_rot @ U_q) @ (K_rot @ U_k)^T
    where Q_rot and K_rot are Q and K AFTER RoPE.
    Only U_q and U_k (float32) are trainable.
    o_proj is NOT wrapped in no_grad so gradients flow through to U_q/U_k.
    """
    def __init__(self, original_attn, rank: int = 4):
        super().__init__()
        self.original_attn    = original_attn
        self.config           = getattr(original_attn, "config", None)
        
        # Robustly fetch attributes that changed location across transformers versions
        self.num_heads        = getattr(original_attn, "num_heads", None) or self.config.num_attention_heads
        self.num_kv_heads     = getattr(original_attn, "num_key_value_heads", None) or self.config.num_key_value_heads
        self.hidden_size      = getattr(original_attn, "hidden_size", None) or self.config.hidden_size
        self.head_dim         = getattr(original_attn, "head_dim", None) or (self.hidden_size // self.num_heads)
        self.num_kv_groups    = self.num_heads // self.num_kv_heads
        
        self.expected_tuple_len = None

        # float32 trainable parameters (adapter weights)
        self.U_q = nn.Parameter(torch.zeros(self.num_heads, self.head_dim, rank))
        self.U_k = nn.Parameter(torch.zeros(self.num_heads, self.head_dim, rank))
        nn.init.normal_(self.U_q, std=0.02)
        nn.init.normal_(self.U_k, std=0.02)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        position_ids=None,
        past_key_value=None,
        output_attentions=False,
        use_cache=False,
        cache_position=None,
        position_embeddings=None,   # (cos, sin) passed by newer transformers versions
        **kwargs,
    ):
        # Run a dummy pass of the original attention exactly once to see what tuple length the 
        # user's specific version of `transformers` expects us to return. 
        # This permanently solves 'too many values to unpack' errors across different library versions.
        if self.expected_tuple_len is None:
            with torch.no_grad():
                dummy_out = self.original_attn(
                    hidden_states=hidden_states,
                    attention_mask=attention_mask,
                    position_ids=position_ids,
                    past_key_value=past_key_value,
                    output_attentions=output_attentions,
                    use_cache=use_cache,
                    cache_position=cache_position,
                    position_embeddings=position_embeddings,
                    **kwargs
                )
                self.expected_tuple_len = len(dummy_out) if isinstance(dummy_out, tuple) else 1

        B, T, _ = hidden_states.shape

        # ------------------------------------------------------------------
        # 1. Frozen QKV projections
        # ------------------------------------------------------------------
        with torch.no_grad():
            Q = self.original_attn.q_proj(hidden_states)  # (B, T, H*D)
            K = self.original_attn.k_proj(hidden_states)  # (B, T, kv_H*D)
            V = self.original_attn.v_proj(hidden_states)  # (B, T, kv_H*D)

        # Reshape to (B, heads, T, head_dim)
        Q = Q.view(B, T, self.num_heads,    self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        # ------------------------------------------------------------------
        # 2. Apply RoPE to Q and K (frozen op)
        # ------------------------------------------------------------------
        with torch.no_grad():
            if position_embeddings is not None:
                # Newer transformers: (cos, sin) are passed in directly
                cos, sin = position_embeddings
            else:
                # Older transformers: compute inside from position_ids
                cos, sin = self.original_attn.rotary_emb(V, position_ids)
            Q, K = apply_rotary_pos_emb(Q, K, cos.float(), sin.float())

            # GQA: expand K and V to match num_heads
            if self.num_kv_groups > 1:
                K = K.repeat_interleave(self.num_kv_groups, dim=1)
                V = V.repeat_interleave(self.num_kv_groups, dim=1)

        # Detach — gradients come only through U_q and U_k
        Q = Q.detach().float()   # (B, H, T, D) float32
        K = K.detach().float()
        V = V.detach().float()

        # ------------------------------------------------------------------
        # 3. Compute Delta — the ONLY differentiable path
        # ------------------------------------------------------------------
        A     = torch.einsum('bhtd,hdr->bhtr', Q, self.U_q)         # (B,H,T,rank)
        Bm    = torch.einsum('bhtd,hdr->bhtr', K, self.U_k)         # (B,H,T,rank)
        delta = torch.matmul(A, Bm.transpose(-1, -2))                # (B,H,T,T) — has grad

        # ------------------------------------------------------------------
        # 4. Frozen baseline scores
        # ------------------------------------------------------------------
        with torch.no_grad():
            S      = torch.matmul(Q, K.transpose(-1, -2)) / (self.head_dim ** 0.5)
            causal = torch.tril(torch.ones(T, T, device=device)).bool()
            S      = S.masked_fill(~causal, torch.finfo(S.dtype).min)

        # ------------------------------------------------------------------
        # 5. Inject Delta
        # ------------------------------------------------------------------
        S_prime = S + delta
        if attention_mask is not None:
            S_prime = S_prime + attention_mask.float()
        attn_weights = F.softmax(S_prime, dim=-1)                    # (B,H,T,T)

        # ------------------------------------------------------------------
        # 6. Weighted sum, merge heads
        # ------------------------------------------------------------------
        attn_output = torch.matmul(attn_weights, V)                  # (B,H,T,D)
        attn_output = attn_output.transpose(1, 2).contiguous()       # (B,T,H,D)
        attn_output = attn_output.view(B, T, self.num_heads * self.head_dim)  # (B,T,C)

        # Cast back to the model's dtype before o_proj
        attn_output = attn_output.to(dtype=hidden_states.dtype)

        # ------------------------------------------------------------------
        # 7. Output projection — NO no_grad: gradients must flow through here
        # ------------------------------------------------------------------
        attn_output = self.original_attn.o_proj(attn_output)         # (B,T,hidden)

        # Pad the output tuple to the exact length expected by the caller's version of transformers
        return (attn_output, None, None, None)[:self.expected_tuple_len]


def inject_calibration(model, rank=4):
    trainable_params = []
    for layer in model.model.layers:          # LlamaForCausalLM structure
        cal = CalibratedAttention(layer.self_attn, rank=rank).to(device)
        layer.self_attn = cal
        trainable_params.extend([cal.U_q, cal.U_k])
    return trainable_params


trainable_params = inject_calibration(student_model, rank=4)
print(f"Injection complete.")
print(f"Trainable scalars: {sum(p.numel() for p in trainable_params):,}  (U_q + U_k across all layers)")

Injection complete.
Trainable scalars: 393,216  (U_q + U_k across all layers)


## Step 7 — Gradient Check

In [8]:
review_chk, _ = TRAIN_REVIEWS[0]
inputs_chk = tokenizer(build_student_prompt(review_chk), return_tensors="pt").to(device)
out_chk = student_model(**inputs_chk, output_hidden_states=True)
hs_chk  = out_chk.hidden_states[-1][:, -1, :]

assert hs_chk.requires_grad, "❌ Gradient check failed!"
print("✅ Gradient check passed — U_q and U_k are in the computation graph.")

✅ Gradient check passed — U_q and U_k are in the computation graph.


## Step 8 — Build Training Pairs (with averaged teacher targets)

In [9]:
def get_teacher_targets(review):
    """
    Run teacher with BOTH demo orderings and average.
    Returns (avg_hidden_state, avg_probability_distribution).
    """
    fwd_inputs = tokenizer(build_teacher_prompt(DEMOS_FORWARD,  review), return_tensors="pt").to(device)
    rev_inputs = tokenizer(build_teacher_prompt(DEMOS_REVERSED, review), return_tensors="pt").to(device)
    with torch.no_grad():
        out_fwd  = teacher_model(**fwd_inputs, output_hidden_states=True)
        out_rev  = teacher_model(**rev_inputs, output_hidden_states=True)
        hs_avg   = ((out_fwd.hidden_states[-1][:, -1, :] + out_rev.hidden_states[-1][:, -1, :]) / 2).detach()
        p_avg    = ((F.softmax(out_fwd.logits[:, -1, :].float(), dim=-1) +
                     F.softmax(out_rev.logits[:, -1, :].float(), dim=-1)) / 2).detach()
    return hs_avg, p_avg


def build_pairs(reviews_with_labels):
    pairs = []
    for review, true_label in reviews_with_labels:
        s_inputs  = tokenizer(build_student_prompt(review), return_tensors="pt").to(device)
        t_hs, t_p = get_teacher_targets(review)
        pairs.append((s_inputs, t_hs, t_p, true_label))
    return pairs


print("Building training pairs (teacher runs 2× per review for bias cancellation)...")
train_pairs = build_pairs(TRAIN_REVIEWS)
print("Building test pairs...")
test_pairs  = build_pairs(TEST_REVIEWS)
print("Done.")

Building training pairs (teacher runs 2× per review for bias cancellation)...
Building test pairs...
Done.


## Step 9 — Results BEFORE Training

In [10]:
print(f"\n{'='*65}")
print("TEST SET — BEFORE TRAINING")
print(f"{'='*65}")

for review, true_label in TEST_REVIEWS:
    t_label, t_pp, t_pn = predict_teacher_averaged(review)
    b_label, b_pp, b_pn = baseline_preds[review]

    t_mark = "✅" if t_label == true_label else "❌"
    b_mark = "✅" if b_label == true_label else "❌"

    print(f"\nReview : \"{review}\"  (True: {true_label})")
    print(f"  Teacher with demonstrations    → {t_label:8s}  [P(Pos)={t_pp:.3f}, P(Neg)={t_pn:.3f}]  {t_mark}")
    print(f"  Student (no demonstrations)   → {b_label:8s}  [P(Pos)={b_pp:.3f}, P(Neg)={b_pn:.3f}]  {b_mark}")


TEST SET — BEFORE TRAINING

Review : "Absolutely love it, the best thing I have bought all year."  (True: Positive)
  Teacher with demonstrations    → Positive  [P(Pos)=0.769, P(Neg)=0.201]  ✅
  Student (no demonstrations)   → Positive  [P(Pos)=0.993, P(Neg)=0.005]  ✅

Review : "Great quality for the price, very impressed with the finish."  (True: Positive)
  Teacher with demonstrations    → Positive  [P(Pos)=0.652, P(Neg)=0.321]  ✅
  Student (no demonstrations)   → Positive  [P(Pos)=0.984, P(Neg)=0.012]  ✅

Review : "Exactly what I needed, works like a charm every single time."  (True: Positive)
  Teacher with demonstrations    → Positive  [P(Pos)=0.709, P(Neg)=0.261]  ✅
  Student (no demonstrations)   → Positive  [P(Pos)=0.991, P(Neg)=0.008]  ✅

Review : "Fantastic product, arrived quickly and in perfect condition."  (True: Positive)
  Teacher with demonstrations    → Positive  [P(Pos)=0.818, P(Neg)=0.159]  ✅
  Student (no demonstrations)   → Positive  [P(Pos)=0.987, P(Neg)=0.010]  

## Step 10 — Training

In [11]:
ALPHA  = 0.5
LAMBDA = 1e-3

optimizer = optim.Adam(trainable_params, lr=1e-3)

epochs = 300
print(f"Training for {epochs} epochs...")
print(f"  Loss = MSE + {ALPHA}×KL + {LAMBDA}×L2  |  grad clip = 1.0\n")

for epoch in range(epochs):
    optimizer.zero_grad()
    epoch_loss = 0.0

    for s_inputs, t_hs, t_probs, _ in train_pairs:
        out      = student_model(**s_inputs, output_hidden_states=True)
        s_hs     = out.hidden_states[-1][:, -1, :].float()
        s_logits = out.logits[:, -1, :].float()

        mse_loss = F.mse_loss(s_hs, t_hs.float())
        kl_loss  = F.kl_div(F.log_softmax(s_logits, dim=-1), t_probs, reduction='batchmean')
        l2_reg   = sum(p.norm() for p in trainable_params)

        epoch_loss += mse_loss + ALPHA * kl_loss + LAMBDA * l2_reg

    epoch_loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
    optimizer.step()

    if (epoch + 1) % 60 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:03d} | Avg Loss: {epoch_loss.item()/len(train_pairs):.4f}")

print("\nTraining Complete!")

Training for 300 epochs...
  Loss = MSE + 0.5×KL + 0.001×L2  |  grad clip = 1.0

Epoch 001 | Avg Loss: 1.1464


KeyboardInterrupt: 

## Step 11 — Results AFTER Training

In [ ]:
print(f"\n{'='*65}")
print("TEST SET — AFTER TRAINING")
print(f"{'='*65}")

for review, true_label in TEST_REVIEWS:
    b_label, b_pp, b_pn = baseline_preds[review]
    s_label, s_pp, s_pn = predict(student_model, build_student_prompt(review))

    b_mark = "✅" if b_label == true_label else "❌"
    s_mark = "✅" if s_label == true_label else "❌"

    print(f"\nReview : \"{review}\"  (True: {true_label})")
    print(f"  Student (no demonstrations)    → {b_label:8s}  [P(Pos)={b_pp:.3f}, P(Neg)={b_pn:.3f}]  {b_mark}")
    print(f"  Student trained by teacher     → {s_label:8s}  [P(Pos)={s_pp:.3f}, P(Neg)={s_pn:.3f}]  {s_mark}")

## Step 12 — Accuracy Summary

In [ ]:
def accuracy(preds, label):
    pos_c = pos_t = neg_c = neg_t = 0
    for (review, true_label), (pred_label, _, _) in zip(TEST_REVIEWS, preds):
        if true_label == "Positive":
            pos_t += 1
            if pred_label == true_label: pos_c += 1
        else:
            neg_t += 1
            if pred_label == true_label: neg_c += 1
    total = pos_t + neg_t
    correct = pos_c + neg_c
    print(f"  {label}")
    print(f"    Positive : {pos_c}/{pos_t}  ({100*pos_c/pos_t:.0f}%)")
    print(f"    Negative : {neg_c}/{neg_t}  ({100*neg_c/neg_t:.0f}%)")
    print(f"    Overall  : {correct}/{total}  ({100*correct/total:.0f}%)")
    return correct / total


teacher_preds = [predict_teacher_averaged(r) for r, _ in TEST_REVIEWS]
student_preds = [predict(student_model, build_student_prompt(r)) for r, _ in TEST_REVIEWS]
base_preds    = [baseline_preds[r] for r, _ in TEST_REVIEWS]

print(f"\n{'='*65}")
print("ACCURACY SUMMARY — TEST SET (10 reviews: 5 pos / 5 neg)")
print(f"{'='*65}")
t_acc = accuracy(teacher_preds, "Teacher with demonstrations")
print()
b_acc = accuracy(base_preds,    "Student (untrained, 0-shot)")
print()
s_acc = accuracy(student_preds, "Student (trained by teacher, 0-shot)")
print(f"""
  Net improvement from distillation : {(s_acc - b_acc)*100:+.0f}%
  Remaining gap to teacher          : {(t_acc - s_acc)*100:+.0f}%
""")

## Step 13 — Hidden State Metrics

In [ ]:
def eval_hidden(pairs, label):
    total_mse = total_cos = 0.0
    with torch.no_grad():
        for s_inputs, t_hs, _, _ in pairs:
            out  = student_model(**s_inputs, output_hidden_states=True)
            s_hs = out.hidden_states[-1][:, -1, :].float()
            total_mse += F.mse_loss(s_hs, t_hs.float()).item()
            total_cos += F.cosine_similarity(s_hs, t_hs.float()).item()
    n = len(pairs)
    print(f"  {label:10s} | Avg MSE: {total_mse/n:.4f} | Avg Cosine Sim: {total_cos/n:.4f}")

print("Hidden State Metrics (after training):")
eval_hidden(train_pairs, "Train")
eval_hidden(test_pairs,  "Test")